# 04 — LLM Client: Groq / OpenRouter (OpenAI-Compatible)

**Project:** ADV_ML_02 — Egyptian Real Estate Chatbot

This notebook builds a fully-functional chatbot session using **any
OpenAI-compatible API** (Groq, OpenRouter, or a custom endpoint). It is
independent of `03_Gemini_Function_Calling.ipynb` and shares the same
`ChatSession`, tools, and system prompt so you can compare both backends
side-by-side.

---

---
## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import json, time, pprint
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 20)

print('Libraries loaded successfully.')
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
# ── Load environment variables ─────────────────────────────────────────────
from src.chatbot.openai_client import load_env_openai

_ENV_OK = False
_ENV_ERROR = ''

try:
    _API_KEY, _MODEL_NAME, _PROVIDER, _BASE_URL = load_env_openai(
        env_path=PROJECT_ROOT / '.env'
    )
    _ENV_OK = True
    print(f'Provider : {_PROVIDER}')
    print(f'Model    : {_MODEL_NAME}')
    print(f'Endpoint : {_BASE_URL}')
    print('API key  : [loaded, not shown]')
except EnvironmentError as e:
    _ENV_ERROR = str(e)
    print(f'[ENV NOT SET] {_ENV_ERROR}')
    print('-> FakeLLMClient tests will still run; live API tests are skipped.')

In [ ]:
# ── httpx version check ────────────────────────────────────────────────────
import httpx
print('httpx version:', httpx.__version__)

---
## 2. OpenAICompatibleClient

`OpenAICompatibleClient` (in `src/chatbot/openai_client.py`) implements the
same `LLMClient` abstract interface as `GeminiClient`.

Key design points:

| Feature | Detail |
|---|---|
| Transport | `httpx` (no `openai` package required) |
| Auth | `Authorization: Bearer <key>` header |
| Retry | 3 attempts, exponential backoff (2 s, 4 s) on 429/503 |
| OpenRouter | Adds `HTTP-Referer` + `X-Title` headers automatically |
| Tool format | Wraps each declaration as `{"type":"function","function":decl}` |
| Tool response | Sends as `{"role":"tool", "tool_call_id":..., "content":json}` |
| API key | Never stored as a readable attribute, never logged |

In [ ]:
from src.chatbot.openai_client import OpenAICompatibleClient
import inspect

# Show the class docstring
print(inspect.getdoc(OpenAICompatibleClient))
print()
print('Retry settings:')
print(f'  _MAX_RETRIES     = {OpenAICompatibleClient._MAX_RETRIES}')
print(f'  _RETRY_BASE_DELAY = {OpenAICompatibleClient._RETRY_BASE_DELAY} s')

---
## 3. Tool Declarations

The chatbot reuses the same six tools from `src/chatbot/tools.py` that were
tested in Notebook 02. `TOOL_DECLARATIONS` is loaded lazily on first import.

`OpenAICompatibleClient._to_openai_tools()` wraps each internal declaration
as `{"type": "function", "function": <decl>}` — the format expected by the
OpenAI chat completions API.

In [ ]:
from src.chatbot.tools import TOOL_DECLARATIONS

print(f'Tools loaded: {len(TOOL_DECLARATIONS)}')
for d in TOOL_DECLARATIONS:
    nparams = len(d.get('parameters', {}).get('properties', {}))
    print(f'  {d["name"]:30s}  {nparams} top-level params')

In [ ]:
# Show how the first tool looks in OpenAI format
# (mirrors what _to_openai_tools() sends to the API)
first_decl = TOOL_DECLARATIONS[0]
oai_format = {'type': 'function', 'function': first_decl}

print('OpenAI-format tool (first):')
print(json.dumps({
    'type': oai_format['type'],
    'function': {
        'name': oai_format['function']['name'],
        'description': oai_format['function']['description'][:120] + '...',
        'parameters': {'type': '...', 'properties': '{ ... }'},
    }
}, indent=2))

---
## 4. Message Format Conversion

The OpenAI API uses a different message role scheme than the internal format.

| Internal | OpenAI |
|---|---|
| `role: "user"`, text part | `{"role": "user", "content": "..."}` |
| `role: "model"`, function_call part | `{"role": "assistant", "tool_calls": [...]}` |
| `role: "user"`, function_response part | `{"role": "tool", "tool_call_id": "...", "content": "..."}` |

The `arguments` field in a tool call response is always a **JSON string** —
`_normalize_response()` calls `json.loads()` to convert it back to a dict.

In [ ]:
# Demo: _to_openai_messages converts the internal format correctly
import uuid

# Build a synthetic two-turn history
fake_call_id = 'call_' + str(uuid.uuid4())[:8]
sample_history = [
    {
        'role': 'user',
        'parts': [{'type': 'text', 'text': 'Show me apartments in Cairo'}]
    },
    {
        'role': 'model',
        'parts': [{
            'type': 'function_call',
            'name': 'search_properties',
            'args': {'filters': {'category': 'rent', 'city': 'Cairo'}},
            'id': fake_call_id,
        }]
    },
    {
        'role': 'user',
        'parts': [{
            'type': 'function_response',
            'name': 'search_properties',
            'id': fake_call_id,
            'response': {'count': 1234, 'results': []},
        }]
    },
]

# Instantiate a dummy client just to call the converter
# (no API key needed — we are not calling generate())
import os
os.environ.setdefault('DUMMY_KEY', 'sk-dummy')

# Build a lightweight dummy client by patching httpx.Client
import unittest.mock as mock
with mock.patch('httpx.Client'):
    dummy_client = OpenAICompatibleClient(
        api_key='sk-dummy',
        model_name='llama-test',
        base_url='http://localhost/v1',
    )

oai_msgs = dummy_client._to_openai_messages(sample_history, system='You are an assistant.')
for m in oai_msgs:
    role = m['role']
    content = m.get('content', '')[:60] if m.get('content') else '<null>'
    tool_calls = m.get('tool_calls', [])
    print(f'role={role:11s}  content={content!r:62s}  tool_calls={len(tool_calls)}')

---
## 5. FakeLLMClient Tests (no API key required)

These tests use `FakeLLMClient` (scripted responses) so they run without any
API key. They verify that `ChatSession` behaves correctly with an
OpenAI-compatible tool loop.

All four tests must **PASS** — if any fails, the core session logic is broken.

In [ ]:
from src.chatbot.chat_engine import FakeLLMClient, ChatSession

PASS_ICON = '[PASS]'
FAIL_ICON = '[FAIL]'

def run_test(name, fn):
    try:
        fn()
        print(f'{PASS_ICON} {name}')
        return True
    except Exception as exc:
        print(f'{FAIL_ICON} {name}: {exc}')
        return False

In [ ]:
# ── Test 1: Basic tool loop + state merge ─────────────────────────────────
def test_basic_loop():
    client = FakeLLMClient([
        {'tool_calls': [{'name': 'search_properties',
                          'args': {'filters': {'category': 'buy', 'city': 'Cairo'}}}]},
        {'text': 'Found 500 properties for sale in Cairo.'},
    ])
    session = ChatSession(session_id='t1', llm_client=client)
    result = session.send('Show me apartments for sale in Cairo')

    assert result['reply'] == 'Found 500 properties for sale in Cairo.',         f'Unexpected reply: {result["reply"]}'
    assert len(result['tool_calls']) == 1
    assert result['tool_calls'][0]['name'] == 'search_properties'
    assert session.state.get('city') == 'Cairo', f'State: {session.state}'

t1_ok = run_test('Basic tool loop + state merge', test_basic_loop)

In [ ]:
# ── Test 2: Tool error is handled gracefully ──────────────────────────────
def test_tool_error():
    client = FakeLLMClient([
        {'tool_calls': [{'name': 'nonexistent_tool', 'args': {}}]},
        {'text': 'I could not complete that request.'},
    ])
    session = ChatSession(session_id='t2', llm_client=client)
    result = session.send('Do something invalid')

    # The tool should have returned an error dict, not raised
    assert result['reply'] == 'I could not complete that request.'
    tc = result['tool_calls'][0]
    assert not tc['ok'], f'Expected ok=False, got: {tc}'
    assert 'error' in tc['result']

t2_ok = run_test('Tool error handled gracefully', test_tool_error)

In [ ]:
# ── Test 3: Five-round cap is enforced ────────────────────────────────────
def test_round_cap():
    # Simulate a model that always calls a tool — should stop after 5 rounds
    script = [
        {'tool_calls': [{'name': 'search_properties',
                          'args': {'filters': {'category': 'rent'}}}]}
        for _ in range(5)
    ] + [{'text': 'Summary after cap.'}]

    client = FakeLLMClient(script)
    session = ChatSession(session_id='t3', llm_client=client)
    result = session.send('Keep searching forever')

    assert result['reply'] == 'Summary after cap.'
    assert len({tc['round'] for tc in result['tool_calls']}) <= 5,         'More than 5 tool rounds used'

t3_ok = run_test('Five-round cap enforced', test_round_cap)

In [ ]:
# ── Test 4: reset() clears state and history ──────────────────────────────
def test_reset():
    client = FakeLLMClient([
        {'tool_calls': [{'name': 'search_properties',
                          'args': {'filters': {'category': 'buy', 'city': 'Giza'}}}]},
        {'text': 'Found results.'},
        # Second send after reset
        {'text': 'Clean start.'},
    ])
    session = ChatSession(session_id='t4', llm_client=client)
    session.send('Find apartments in Giza')

    assert session.state.get('city') == 'Giza'

    session.reset()
    assert session.state == {}, f'State not cleared: {session.state}'
    assert session._history == []
    assert session.metrics == []

    result2 = session.send('Hello again')
    assert result2['reply'] == 'Clean start.'

t4_ok = run_test('reset() clears state and history', test_reset)

print()
all_ok = all([t1_ok, t2_ok, t3_ok, t4_ok])
print('All tests PASSED' if all_ok else 'SOME TESTS FAILED')

---
## 6. Build the OpenAICompatibleClient (live API)

The cell below instantiates `OpenAICompatibleClient` with your real API key.
It is **skipped automatically** if `_ENV_OK` is `False` (key not set).

> **Never print or log the API key.** The client stores it only inside the
> `httpx.Client` Authorization header and never as a readable attribute.

In [ ]:
if _ENV_OK:
    from src.chatbot.openai_client import OpenAICompatibleClient

    llm_client = OpenAICompatibleClient(
        api_key=_API_KEY,
        model_name=_MODEL_NAME,
        base_url=_BASE_URL,
        provider=_PROVIDER,
        timeout=60.0,
    )
    print(f'Client ready: provider={_PROVIDER!r}  model={_MODEL_NAME!r}')
else:
    print(f'[SKIPPED] API key not set: {_ENV_ERROR}')

---
## 7. Smoke Test — single turn

Sends one query to the real API and verifies that:
1. The model calls `search_properties` (or `resolve_location` first).
2. A non-empty text reply is returned.
3. Token usage is reported.

Skipped if API key is not configured.

In [ ]:
if _ENV_OK:
    import time as _time

    session_smoke = ChatSession(
        session_id='smoke-01',
        llm_client=llm_client,
    )

    print('Sending: "Show me apartments for rent in Giza"')
    t0 = _time.perf_counter()
    smoke = session_smoke.send('Show me apartments for rent in Giza')
    elapsed = (_time.perf_counter() - t0) * 1000

    print(f'Turn time : {elapsed:.0f} ms')
    print(f'Tool calls: {len(smoke["tool_calls"])}')
    for tc in smoke['tool_calls']:
        status = 'ok' if tc['ok'] else 'ERROR'
        print(f'  [{status}] {tc["name"]}  ({tc["ms"]:.0f} ms)')
    print(f'Usage     : {smoke["usage"]}')
    print()
    print('Reply:')
    print(smoke['reply'][:600])
else:
    print('[SKIPPED]')

In [ ]:
if _ENV_OK:
    # Show suggestions from suggest_next_question
    print('Suggestions:', smoke.get('suggestions', []))
    print('Properties returned:', len(smoke.get('properties', [])))
    print('Grounding warn:', smoke.get('grounding_warn'))
else:
    print('[SKIPPED]')

---
## 8. Multi-turn Conversation

Two complete multi-turn conversations — one in **English**, one in **Arabic**.
Each starts fresh (new `ChatSession`).

> The model is expected to:
> - Call `resolve_location` before `search_properties` when a place name is mentioned.
> - Progressively narrow results across turns.
> - Respond in the same language as the user.

In [ ]:
if _ENV_OK:
    session_en = ChatSession(
        session_id='multi-en',
        llm_client=llm_client,
    )

    turns_en = [
        'I am looking for an apartment to buy in New Cairo',
        'Budget around 3 million EGP',
        'I want 3 bedrooms',
    ]

    print('=== English conversation ===')
    for msg in turns_en:
        print(f'\nUser: {msg}')
        resp = session_en.send(msg)
        print(f'Bot : {resp["reply"][:400]}')
        calls = [tc["name"] for tc in resp["tool_calls"]]
        if calls:
            print(f'     [Tools: {calls}]')
else:
    print('[SKIPPED]')

In [ ]:
if _ENV_OK:
    session_ar = ChatSession(
        session_id='multi-ar',
        llm_client=llm_client,
    )

    turns_ar = [
        'عايز شقة للإيجار في مدينة نصر',
        'ميزانيتي 8000 جنيه في الشهر',
        'عايزها 2 أوضة نوم',
    ]

    print('=== Arabic conversation ===')
    for msg in turns_ar:
        print(f'\nUser: {msg}')
        resp = session_ar.send(msg)
        print(f'Bot : {resp["reply"][:400]}')
        calls = [tc["name"] for tc in resp["tool_calls"]]
        if calls:
            print(f'     [Tools: {calls}]')
else:
    print('[SKIPPED]')

---
## 9. Zero-result → relax_filters

When `search_properties` returns 0 results the model should call `relax_filters`
and show the user which constraint was loosened.

In [ ]:
if _ENV_OK:
    session_relax = ChatSession(
        session_id='relax-01',
        llm_client=llm_client,
    )

    print('Sending a query with very tight filters ...')
    resp = session_relax.send(
        'Show me villas for sale in Alexandria '
        'with 10 bedrooms, price under 500,000 EGP, '
        'must have swimming pool, gym, and rooftop garden'
    )
    print(f'Tool calls: {[tc["name"] for tc in resp["tool_calls"]]}')
    print()
    print('Reply:')
    print(resp['reply'][:600])
else:
    print('[SKIPPED]')

---
## 10. Error Handling

Demonstrates what happens when an invalid API key is used.
The `OpenAICompatibleClient` raises `httpx.HTTPStatusError` (401 Unauthorized)
rather than hanging or returning silently.

In [ ]:
import httpx

print('Testing invalid API key ...')
try:
    bad_client = OpenAICompatibleClient(
        api_key='sk-invalid-key-00000',
        model_name=_MODEL_NAME if _ENV_OK else 'llama-3.3-70b-versatile',
        base_url=_BASE_URL if _ENV_OK else 'https://api.groq.com/openai/v1',
        provider=_PROVIDER if _ENV_OK else 'groq',
        timeout=10.0,
    )
    bad_session = ChatSession(session_id='err-test', llm_client=bad_client)
    bad_session.send('Hello')
    print('[UNEXPECTED] No error raised.')
except httpx.HTTPStatusError as e:
    print(f'[EXPECTED] httpx.HTTPStatusError: status={e.response.status_code}')
except Exception as e:
    print(f'[EXPECTED] {type(e).__name__}: {str(e)[:120]}')

In [ ]:
# Demonstrate retry logic: show the constants
print('Retry configuration:')
print(f'  MAX_RETRIES      = {OpenAICompatibleClient._MAX_RETRIES}')
print(f'  RETRY_BASE_DELAY = {OpenAICompatibleClient._RETRY_BASE_DELAY} s')
print()
print('Retry schedule for a transient 429 error:')
for attempt in range(OpenAICompatibleClient._MAX_RETRIES - 1):
    delay = OpenAICompatibleClient._RETRY_BASE_DELAY * (2 ** attempt)
    print(f'  Attempt {attempt + 1}: wait {delay:.1f} s before retry')
print(f'  Attempt {OpenAICompatibleClient._MAX_RETRIES}: raise (no more retries)')

---
## 11. Session Metrics

Every `ChatSession` accumulates a `metrics` list with one entry per turn.
This cell aggregates all test sessions and exports to CSV.

In [ ]:
import os

all_metrics = []

if _ENV_OK:
    for sess in [session_smoke, session_en, session_ar, session_relax]:
        all_metrics.extend(sess.metrics)

if all_metrics:
    df_metrics = pd.DataFrame(all_metrics)
    display(df_metrics[['session_id', 'turn', 'tool_calls',
                         'properties_returned', 'total_tokens', 'turn_ms']])

    out_dir = PROJECT_ROOT / 'data' / 'processed' / 'chat_bot'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'engine_metrics_groq.csv'
    df_metrics.to_csv(out_path, index=False)
    print(f'Metrics saved: {out_path}')
else:
    print('[SKIPPED] No live sessions to report (API key not set).')

---
## 12. Findings & Comparison vs Gemini

### Architecture differences

| Aspect | Gemini (`03_Gemini_Function_Calling`) | Groq/OpenRouter (`04_LLMClient_Groq`) |
|---|---|---|
| Transport | `google-genai` SDK | `httpx` (raw REST) |
| Tool format | `FunctionDeclaration` objects | `{"type":"function","function":{...}}` |
| Function response | `FunctionResponse` Content part | `{"role":"tool","tool_call_id":...}` |
| Schema compaction | `_compact_decl` / `_select_tools` | Full schema sent every turn |
| Multi-tool calls | One `tool_calls` part list | `message.tool_calls` list |
| Args format | `fc.args` (dict) | `fn.arguments` (JSON string → `json.loads`) |
| Token counting | `usage_metadata` | `usage.completion_tokens` |
| AFC | Must be disabled explicitly | Not applicable (manual loop) |

### Schema size note

Gemini applies `_select_tools` and `_compact_decl` to reduce the per-request
payload size. The OpenAI client sends the full schema every turn. For Groq
this is fine (generous token limits); OpenRouter may charge more tokens.
A future optimisation could port the compaction logic to `OpenAICompatibleClient`.

### Known issues in `tools.py` (do NOT fix — for reference only)

1. `suggest_next_question` uses `.apply()` lambda — slow for large filter sets.
2. `relax_filters` reduces `bedrooms` silently without warning the user.
3. 15 district names appear under multiple cities; `resolve_location` may be ambiguous.
4. `_apply_filters` for `district` has a stray unused `col = df.get(...)` line (~line 217).
5. `TOOL_DECLARATIONS` is populated lazily — first import is slower (cold-start).

### Groq model notes

- `llama-3.3-70b-versatile` handles Egyptian Arabic well.
- `mixtral-8x7b-32768` has a 32k context window — useful for long conversations.
- `llama-3.1-8b-instant` is faster but less reliable on complex tool-call chains.
- OpenRouter models may require `tool_choice` adjustment; some don't support parallel calls.